In [9]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q tensorflow tensorflowjs matplotlib
import os, tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

tf.get_logger().setLevel('ERROR')

BASE_DIR = '/content/drive/MyDrive'
DATASET_PATH = f'{BASE_DIR}/dataset'
MODEL_PATH   = f'{BASE_DIR}/plant_model.keras'
TFJS_PATH    = f'{BASE_DIR}/plant_model_js'

os.makedirs(TFJS_PATH, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:

IMG_SIZE, BATCH, EPOCHS_NEW, EPOCHS_FINE = 224, 32, 10, 5

datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
    zoom_range=0.2, horizontal_flip=True
)

train_gen = datagen.flow_from_directory(DATASET_PATH, target_size=(IMG_SIZE, IMG_SIZE),
                                        batch_size=BATCH, class_mode='categorical',
                                        subset='training', shuffle=True)
val_gen = datagen.flow_from_directory(DATASET_PATH, target_size=(IMG_SIZE, IMG_SIZE),
                                      batch_size=BATCH, class_mode='categorical',
                                      subset='validation', shuffle=False)

num_classes = len(train_gen.class_indices)

Found 7674 images belonging to 5 classes.
Found 1917 images belonging to 5 classes.


In [11]:
def build_model(num_classes):
    base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    x = GlobalAveragePooling2D()(base.output)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    out = Dense(num_classes, activation='softmax')(x)
    model = Model(base.input, out)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

if os.path.exists(MODEL_PATH):
    model = load_model(MODEL_PATH)
    for layer in model.layers[-20:]:
        if not isinstance(layer, Dropout): layer.trainable = True
    epochs = EPOCHS_FINE
else:
    model = build_model(num_classes)
    epochs = EPOCHS_NEW

callbacks = [
    ModelCheckpoint(MODEL_PATH, save_best_only=True, monitor='val_accuracy', mode='max'),
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
]

history = model.fit(train_gen, validation_data=val_gen, epochs=epochs, callbacks=callbacks)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
240/240 ━━━━━━━━━━━━━━━━━━━━ 1860s 8s/step - accuracy: 0.9942 - loss: 0.0249 - val_accuracy: 0.6682 - val_loss: 5.5907
Epoch 2/5
240/240 ━━━━━━━━━━━━━━━━━━━━ 581s 2s/step - accuracy: 0.9925 - loss: 0.0338 - val_accuracy: 0.6234 - val_loss: 7.0619
Epoch 3/5
240/240 ━━━━━━━━━━━━━━━━━━━━ 600s 2s/step - accuracy: 0.9965 - loss: 0.0132 - val_accuracy: 0.9259 - val_loss: 0.6252
Epoch 4/5
240/240 ━━━━━━━━━━━━━━━━━━━━ 588s 2s/step - accuracy: 0.9952 - loss: 0.0182 - val_accuracy: 0.9786 - val_loss: 0.1509
Epoch 5/5
240/240 ━━━━━━━━━━━━━━━━━━━━ 591s 2s/step - accuracy: 0.9982 - loss: 0.0076 - val_accuracy: 0.9525 - val_loss: 0.5107


In [12]:
# Cài đặt thư viện cần thiết cho việc lưu .h5 và convert
!pip install -q tensorflowjs h5py

import tensorflowjs as tfjs
import os
import tensorflow as tf

# --- 1. KIỂM TRA BIẾN 'model' ---
try:
    # Kiểm tra xem biến 'model' có tồn tại từ Cell 3 không
    _ = model.name
    print("✅ Đã tìm thấy 'model' trong bộ nhớ.")
except NameError:
    print("⛔ LỖI: Không tìm thấy biến 'model'.")
    print("Vui lòng chạy lại Cell 3 (train model) trước khi chạy cell này.")
    raise

# --- 2. ĐỊNH NGHĨA ĐƯỜNG DẪN ---
# Các biến này đã được định nghĩa ở Cell 1
# MODEL_PATH = f'{BASE_DIR}/plant_model.keras'
# TFJS_PATH  = f'{BASE_DIR}/plant_model_js'

# Đường dẫn cho file .h5 trung gian (tạm thời)
H5_PATH_TEMP = f'{BASE_DIR}/plant_model_temp.h5'

# --- 3. LƯU SANG ĐỊNH DẠNG .H5 (TỪ 'model' TRONG BỘ NHỚ) ---
print(f"Đang lưu model (từ bộ nhớ) sang định dạng .h5 tại: {H5_PATH_TEMP}")
model.save(H5_PATH_TEMP) # Cần thư viện h5py

# --- 4. LƯU FILE .KERAS (ĐỂ DÙNG SAU NÀY) ---
print(f"Đang lưu model (từ bộ nhớ) sang định dạng .keras tại: {MODEL_PATH}")
model.save(MODEL_PATH) # Lưu file .keras chính thức

# --- 5. CHUẨN BỊ THƯ MỤC OUTPUT ---
print(f"Đang chuẩn bị thư mục output: {TFJS_PATH}")
!rm -rf {TFJS_PATH}
os.makedirs(TFJS_PATH, exist_ok=True)

# --- 6. CHẠY CONVERTER TRÊN FILE .H5 TẠM ---
# Chú ý: input_format bây giờ là 'keras' (dùng cho .h5)
print("Đang chạy tensorflowjs_converter...")
!tensorflowjs_converter --input_format=keras \
                       {H5_PATH_TEMP} \
                       {TFJS_PATH}

# --- 7. GHI FILE LABELS ---
print("Đang ghi file labels.js...")
try:
    # Thử dùng biến train_gen từ Cell 2
    with open(os.path.join(TFJS_PATH, 'labels.js'), 'w') as f:
        f.write(f"const CLASS_NAMES = {list(train_gen.class_indices.keys())};")
    print("✅ Đã ghi labels.js từ 'train_gen'.")
except NameError:
    # Nếu chạy cell này độc lập, train_gen không tồn tại
    print("⚠️ Không tìm thấy biến 'train_gen', đang dùng danh sách dự phòng.")
    class_names_fallback = [
      "Apple___Apple_scab",
      "Apple___Black_rot",
      "Apple___Cedar_apple_rust",
      "Apple___healthy",
      "Blueberry___healthy",
    ] # Lấy từ file script.js của bạn
    with open(os.path.join(TFJS_PATH, 'labels.js'), 'w') as f:
        f.write(f"const CLASS_NAMES = {class_names_fallback};")

print("\n--- HOÀN TẤT ---")
print(f"✅ Đã tạo file Keras: {MODEL_PATH}")
print(f"✅ Đã tạo thư mục TFJS: {TFJS_PATH}")
print("\n🎉 Vui lòng tải thư mục 'plant_model_js' mới lên GitHub và kiểm tra lại trang web.")

✅ Đã tìm thấy 'model' trong bộ nhớ.
Đang lưu model (từ bộ nhớ) sang định dạng .h5 tại: /content/drive/MyDrive/plant_model_temp.h5
Đang lưu model (từ bộ nhớ) sang định dạng .keras tại: /content/drive/MyDrive/plant_model.keras
Đang chuẩn bị thư mục output: /content/drive/MyDrive/plant_model_js
Đang chạy tensorflowjs_converter...
2025-11-07 13:36:32.011292: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762522592.076687   27279 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762522592.097064   27279 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1762522592.171559   27279 computation_placer.cc:177] computation placer already registered. Plea